# 🏥 Phân tích tập dữ liệu chi phí y tế cá nhân

**Nguồn dữ liệu:** [Kaggle - Medical Cost Personal Datasets](https://www.kaggle.com/datasets/mirichoi0218/insurance)


---

In [ ]:
import pandas as pd
from src.preprocessing import load_data, remove_outliers_iqr
from src.analysis import get_correlations, test_smoker_hypothesis
from src.visualization import (
    plot_age_vs_charges, 
    plot_smoker_costs_boxplot, 
    plot_correlation_heatmap, 
    plot_bmi_vs_charges
)

print("Thiết lập thành công.")

In [ ]:
# Tải dữ liệu thô
df_raw = load_data()

---

## ⚙️ 1. Tiền xử lý dữ liệu

### Mục tiêu:
- Đánh giá chất lượng và tính nhất quán của dữ liệu.
- Xử lý các giá trị thiếu và giá trị ngoại lệ bằng các phương pháp thống kê (ví dụ: IQR).
- Tài liệu hóa lý do đằng sau các lựa chọn làm sạch dữ liệu.

### **Tổng quan về tập dữ liệu**

In [ ]:
print(f"Kích thước tập dữ liệu thô: {df_raw.shape}")
df_raw.head()

Tập dữ liệu được sử dụng trong dự án này có chất lượng khá tốt, dữ liệu sạch và có cấu trúc rõ ràng. Bộ dữ liệu bao gồm **1.338 dòng dữ liệu** và **7 thuộc tính**.

#### Các thuộc tính trong tập dữ liệu

| Thuộc tính | Mô tả |
|---|---|
| `age` | Tuổi của người tham gia bảo hiểm |
| `sex` | Giới tính |
| `bmi` | Chỉ số khối cơ thể (Body Mass Index) |
| `children` | Số lượng người phụ thuộc |
| `smoker` | Tình trạng hút thuốc |
| `region` | Khu vực sinh sống tại Mỹ |
| `charges` | Chi phí bảo hiểm y tế cá nhân |

### **Kiểm tra giá trị thiếu (Missing Values)**

In [ ]:
# Kiểm tra giá trị thiếu
print("\nGiá trị thiếu:")
df_raw.isnull().sum()

Tập dữ liệu **không chứa bất kỳ giá trị thiếu nào** ở tất cả các cột.

### **Xử lý giá trị ngoại lệ (Outliers)**



Để xử lý các giá trị ngoại lệ trong cột `charges`, chúng tôi sử dụng phương pháp **IQR (Interquartile Range)**.

Công thức:
- Lower Bound = Q1 − 1.5 × IQR
- Upper Bound = Q3 + 1.5 × IQR

In [ ]:
# Loại bỏ các giá trị ngoại lệ từ cột 'charges' bằng IQR
df_clean = remove_outliers_iqr(df_raw, 'charges')
print(f"Kích thước tập dữ liệu sau khi làm sạch: {df_clean.shape}")

Việc loại bỏ các giá trị ngoại lệ giúp dữ liệu ổn định hơn cho các bước phân tích tiếp theo.

### **Categorical Encoding**


#### 1. **Biến nhị phân (Label Encoding):**
Đối với biến smoker (tình trạng hút thuốc) và sex (giới tính), nhóm thực hiện ánh xạ trực tiếp các nhãn chuỗi sang giá trị số nguyên
#### 2. **Biến đa hình (One-Hot Encoding):**
Đối với biến region (khu vực cư trú), do các giá trị không có tính thứ tự (northeast, northwest, southeast, southwest), nhóm sử dụng kỹ thuật One-Hot Encoding. Kỹ thuật này tạo ra các cột riêng biệt cho từng giá trị, giúp mô hình đánh giá các khu vực một cách bình đẳng.


In [ ]:
# Encode smoker: yes -> 1, no -> 0
df_clean['smoker'] = df_clean['smoker'].map({'yes': 1, 'no': 0})

# Encode sex: female -> 1, male -> 0 
df_clean['sex'] = df_clean['sex'].map({'female': 1, 'male': 0})

# Encode Region
df_clean = pd.get_dummies(df_clean, columns=['region'], drop_first=False, dtype=int)

In [ ]:
df_clean.head(10)

Đối với biến `smoker` (tình trạng hút thuốc) và `sex` (giới tính), nhóm thực hiện ánh xạ trực tiếp các nhãn chuỗi sang giá trị số nguyên:

| Đặc trưng | Giá trị gốc | Giá trị mã hóa |
| --- | --- | --- |
| `smoker` | `no` / `yes` | **0** / **1** |
| `sex` | `male` / `female` | **0** / **1** |


Đối với biến `region` (khu vực cư trú), do các giá trị không có tính thứ tự (northeast, northwest, southeast, southwest), nhóm sử dụng kỹ thuật **One-Hot Encoding**. Kỹ thuật này tạo ra các cột riêng biệt cho từng giá trị, giúp mô hình đánh giá các khu vực một cách bình đẳng.

| Giá trị gốc | region_northeast | region_northwest |region_southeast | region_southwest |
| --- | --- | --- | --- | --- |
| `northeast` | **1** | **0** | **0** | **0** |
| `northwest` | **0** | **1** | **0** | **0** |
| `southeast` | **0** | **0** | **1** | **0** |
| `southwest` | **0** | **0** | **0** | **1** |


---

## 🔍 2. Phân tích dữ liệu khám phá (EDA)

### Mục tiêu:
- Kiểm chứng các giả thuyết chính (ví dụ: người hút thuốc có chi phí y tế cao hơn).
- Phân tích ma trận tương quan giữa các biến.
- Xác định các mẫu và phân khúc trong dữ liệu.

### **Thống kê mô tả**

In [ ]:
# Thống kê mô tả
print("Thống kê mô tả:")
display(df_clean.describe())

1. **Cơ cấu mẫu:**
* Tập dữ liệu bao gồm **1.199 bản ghi** hợp lệ. Độ tuổi trung bình là khoảng **39 tuổi**, dao động từ 18 đến 64 tuổi.
* **Biến Smoker:** Giá trị trung bình của cột `smoker` là **0.115**. Điều này có nghĩa là khoảng **11.5%** đối tượng trong tập dữ liệu là người hút thuốc.


2. **Chỉ số hình thể (BMI):**
* Chỉ số BMI trung bình là **30.1**, nằm ngay ngưỡng phân loại **béo phì** theo tiêu chuẩn quốc tế.
* Giá trị BMI cao nhất lên tới **53.13**, cho thấy có những trường hợp béo phì nghiêm trọng trong tập dữ liệu.


3. **Chi phí y tế (Charges):**
* Chi phí trung bình là **9.927 USD**, tuy nhiên độ lệch chuẩn (`std`) lên tới **7.241 USD**, cho thấy sự biến động cực lớn về chi phí giữa các cá nhân.
* Giá trị trung vị (Median - 50%) là **8.410 USD**, thấp hơn giá trị trung bình (Mean). Điều này gợi ý rằng phân phối của chi phí y tế có hiện tượng **lệch phải**, bị ảnh hưởng bởi một nhóm nhỏ các ca có chi phí điều trị rất cao.


4. **Số lượng con cái (Children):**
* Trung bình mỗi người có khoảng **1 con**. Tuy nhiên, mức 75% vẫn là 2 con, chứng tỏ đa số đối tượng khảo sát có quy mô gia đình nhỏ (từ 0-2 con).



### **Phân tích tương quan**

In [ ]:
# Phân tích tương quan
corr_matrix = get_correlations(df_clean)
print("\nMa trận tương quan:")
display(corr_matrix)

Để hiểu rõ mối quan hệ giữa các biến, ta sử dụng ma trận tương quan Pearson. Kết quả cho thấy các phát hiện quan trọng sau:

| Biến | Tương quan với `charges` | Nhận xét |
| :--- | :---: | :--- |
| **smoker** | **0.6022** | **Tương quan thuận mạnh.** Hút thuốc là yếu tố ảnh hưởng lớn nhất đến chi phí. |
| **age** | **0.4376** | **Tương quan thuận trung bình.** Chi phí có xu hướng tăng dần theo độ tuổi. |
| **children** | 0.0837 | Tương quan thuận rất yếu, ít có tác động trực tiếp đến chi phí. |
| **region_northeast**| 0.0640 | Tương quan rất yếu, gần như không ảnh hưởng. |
| **region_northwest**| 0.0352 | Tương quan rất yếu. |
| **sex** | 0.0244 | Tương quan rất yếu, giới tính không phải là yếu tố quyết định. |
| **region_southeast**| -0.0286 | Tương quan nghịch rất yếu. |
| **bmi** | -0.0665 | Tương quan nghịch rất yếu. Không có mối liên hệ tuyến tính đáng kể trong mẫu này. |
| **region_southwest**| -0.0709 | Tương quan nghịch rất yếu. |

#### Kết luận:
1. **Biến mục tiêu chủ chốt:** `smoker` và `age` là hai đặc trưng (features) quan trọng nhất cần tập trung để dự báo chi phí y tế.
2. **Các biến ít ảnh hưởng:** Giới tính (`sex`) và các yếu tố vùng miền (`region_*`) có mức độ tương quan tuyến tính rất thấp, có thể cân nhắc loại bỏ nếu cần giảm chiều dữ liệu trong các mô hình học máy đơn giản.
3. **Hiện tượng phi tuyến tính:** Biến `bmi` có hệ số tương quan thấp (-0.0665) nhưng có thể tồn tại các mối quan hệ phi tuyến tính hoặc tương tác (interaction effects) với các biến khác (như béo phì kết hợp hút thuốc) mà ma trận tương quan đơn thuần chưa thể hiện hết.

In [ ]:
columns_to_drop = [
    'region_northeast', 
    'region_northwest', 
    'region_southeast', 
    'region_southwest',
    'children',
    'sex'
]

df_clean = df_clean.drop(columns=columns_to_drop)

Dựa trên phân tích ma trận tương quan Pearson, chúng ta tiến hành tối ưu hóa tập dữ liệu bằng cách loại bỏ các đặc trưng có độ tương quan rất thấp với biến mục tiêu `charges`.

#### **Tối ưu hóa dữ liệu (Feature Selection)**

Việc loại bỏ các biến có độ tương quan rất thấp giúp chúng ta làm nổi bật các mối quan hệ then chốt, tránh bị gây nhiễu bởi những yếu tố không đáng kể, từ đó giúp các nhận định và báo cáo insight trở nên sắc bén và trực diện hơn.

#### **Lý do loại bỏ:**
*   **Vùng miền (`region_*`):** Các hệ số tương quan dao động từ -0.07 đến 0.06, cho thấy vị trí địa lý không phải là yếu tố quyết định đến chi phí y tế trong tập dữ liệu này. 
*   **Số lượng con cái (`children`):** Hệ số tương quan rất thấp (0.0837), ảnh hưởng không đáng kể so với các yếu tố khác. 
*   **Giới tính (`sex`):** Hệ số tương quan chỉ đạt 0.0244, cho thấy mức độ chi trả y tế giữa nam và nữ là tương đương nhau. 

Sau khi xử lý, tập dữ liệu sẽ tập trung vào các biến quan trọng nhất là `smoker`, `age` và `bmi`.

### **Kiểm định giả thuyết**

In [ ]:
# Kiểm định giả thuyết: Người hút thuốc vs Chi phí
print("\n--- Kiểm định giả thuyết: Người hút thuốc vs Người không hút thuốc ---")
hypothesis_results = test_smoker_hypothesis(df_clean)
for key, val in hypothesis_results.items():
    print(f"{key}: {val}")

if hypothesis_results['significant']:
    print("\nKết luận: Có sự khác biệt đáng kể về mặt thống kê về chi phí y tế giữa người hút thuốc và người không hút thuốc.")
else:
    print("\nKết luận: Không tìm thấy sự khác biệt đáng kể.")

Để xác nhận mối liên hệ giữa hành vi hút thuốc và chi phí y tế, chúng ta thực hiện kiểm định **Independent T-test** giữa hai nhóm: người hút thuốc (smoker) và người không hút thuốc (non-smoker).

#### Kết quả kiểm định:
* **Chi phí trung bình (Smoker):** $22,014.25
* **Chi phí trung bình (Non-smoker):** $8,355.71
* **Chỉ số T-statistic:** 26.0987
* **P-value:** 3.125e-119
* **Ý nghĩa thống kê:** Có (Significant = True)

#### Kết luận:
1. **Sự chênh lệch đáng kể:** Có sự khác biệt cực kỳ lớn về chi phí y tế giữa hai nhóm. Trung bình, một người hút thuốc phải chi trả cao hơn khoảng **13,658 USD** so với người không hút thuốc.
2. **Độ tin cậy cao:** Với giá trị $P-value$ gần như bằng 0 ($3.12 \times 10^{-119}$), ta có đủ cơ sở để bác bỏ giả thuyết không (H0) và khẳng định rằng việc hút thuốc là một trong những nguyên nhân chính dẫn đến sự gia tăng chi phí y tế trong tập dữ liệu này.
3. **Giá trị phân tích:** Kết quả này củng cố thêm cho hệ số tương quan ($0.6022$) đã tìm thấy ở bước trước, chứng minh rằng `smoker` là biến biến đầu vào quan trọng hàng đầu cho mô hình dự báo.

### **Tổng hợp Insight từ quá trình EDA**
#### Qua các bước thống kê và kiểm định, ta nhận thấy chi phí y tế không chỉ phụ thuộc vào từng biến riêng lẻ mà còn là sự kết hợp giữa chúng.

#### Tương tác giữa Hút thuốc và Chỉ số BMI
Mặc dù chỉ số BMI đứng đơn lẻ không tương quan mạnh với chi phí, nhưng khi kết hợp với yếu tố hút thuốc, ta thấy một hiện tượng đặc biệt:
* Nhóm người **hút thuốc có BMI > 30** (ngưỡng béo phì) có mức chi phí y tế cao vượt trội so với tất cả các nhóm còn lại.
* Điều này gợi ý rằng rủi ro sức khỏe (và chi phí) sẽ tăng theo cấp số nhân khi các yếu tố nguy cơ xuất hiện cùng lúc.

#### Phân tích theo phân khúc độ tuổi
Chi phí y tế có xu hướng tăng ổn định theo độ tuổi đối với người không hút thuốc. Tuy nhiên, đối với người hút thuốc, chi phí này ở mức cao ngay cả ở độ tuổi trẻ (18-25).

#### Kết luận chung của quá trình EDA
1. **Yếu tố quyết định:** Hút thuốc là biến số có tác động mạnh nhất đến chi phí bảo hiểm y tế trong tập dữ liệu này.
2. **Đặc điểm phân phối:** Biến mục tiêu `charges` bị lệch phải, cho thấy phần lớn chi phí nằm ở mức thấp nhưng có một số ít trường hợp chi trả cực cao (thường là người hút thuốc kết hợp béo phì).
3. **Giá trị ứng dụng:** Kết quả phân tích này giúp các đơn vị bảo hiểm xác định rõ các phân khúc khách hàng rủi ro cao để điều chỉnh chính sách phí phù hợp mà chưa cần dùng đến các mô hình máy học phức tạp.

---

## 📊 3. Trực quan hóa dữ liệu theo mục tiêu

### Trọng tâm phân tích:
- **Biểu đồ phân tán**: Mối quan hệ giữa các biến liên tục (Tuổi/BMI vs Chi phí).
- **Biểu đồ hộp**: Phân bổ chi phí giữa các nhóm (Hút thuốc vs Không hút thuốc).

### **Trực quan hóa ma trận tương quan (Heatmap)**

In [ ]:
plot_correlation_heatmap(df_clean)

Bản đồ nhiệt dưới đây cung cấp cái nhìn trực quan về mối liên hệ giữa các đặc trưng số trong tập dữ liệu:

#### Quan sát chủ chốt:
1. **Yếu tố tác động chính:** Biến `smoker` và `age` hiển thị các sắc thái màu nóng (cam), thể hiện mối tương quan thuận rõ rệt với chi phí y tế (`charges`). 
2. **Tính độc lập của các biến:** Biến `bmi` hiển thị các sắc thái màu lạnh, cho thấy chúng ít có mối quan hệ tuyến tính trực tiếp với mức chi phí y tế trong tập dữ liệu này.
3. **Mối quan hệ giữa các biến độc lập:** Không có hiện tượng đa cộng tuyến (multicollinearity) nghiêm trọng giữa các biến dự báo (các hệ số tương quan giữa các biến độc lập đều thấp), điều này rất có lợi cho việc phân tích các tác động riêng lẻ của từng biến lên chi phí.

#### Kết luận phân tích:
Việc trực quan hóa bằng Heatmap giúp xác nhận lại các kết luận từ bảng thống kê: Để quản lý hoặc dự báo chi phí y tế, hai chỉ số quan trọng nhất cần tập trung khai thác là **tình trạng hút thuốc** và **độ tuổi** của đối tượng.

### **Trực quan hóa mối quan hệ giữa Tuổi, Hút thuốc và Chi phí (Scatter Plot)** 


In [ ]:
# Tạo các biểu đồ
plot_age_vs_charges(df_clean)



Biểu đồ Scatter Plot dưới đây thể hiện sự phân hóa rõ rệt về chi phí y tế dựa trên hai yếu tố chủ chốt là độ tuổi và tình trạng hút thuốc.

#### Quan sát chi tiết từ biểu đồ:

1. **Sự phân tách thành các "tầng" chi phí:**
* **Tầng thấp (Nhóm dưới cùng):** Chủ yếu là những người **không hút thuốc (smoker = 0)**. Chi phí y tế của nhóm này tăng rất đều và chậm theo độ tuổi, tạo thành một đường thẳng dốc lên rõ rệt.
* **Tầng trung và cao:** Hầu hết là những người **hút thuốc (smoker = 1)**. Ngay cả ở độ tuổi trẻ (20 tuổi), chi phí của nhóm hút thuốc đã dao động từ 15.000 đến hơn 30.000 USD, cao hơn rất nhiều so với nhóm không hút thuốc cùng độ tuổi.


2. **Tác động của tuổi tác (Age):**
* Cả hai nhóm đều cho thấy xu hướng: **Tuổi càng cao, chi phí y tế càng tăng**. Điều này khẳng định mối tương quan thuận (positive correlation) đã thấy ở bước phân tích ma trận tương quan.
* Tuy nhiên, với người không hút thuốc, sự gia tăng này là tuyến tính và dễ dự báo. Với người hút thuốc, chi phí bị phân tán mạnh hơn rất nhiều.


3. **Sự phân hóa đặc biệt trong phân phối chi phí:**
* Có một khoảng trống lớn về chi phí giữa nhóm không hút thuốc (dưới 15.000 USD) và nhóm hút thuốc rủi ro cao (trên 30.000 USD) ở độ tuổi trẻ.
* Chi phí của người không hút thuốc tập trung thành một dải hẹp và ổn định, trong khi chi phí của người hút thuốc bị phân tán rộng, cho thấy mức độ ảnh hưởng của hành vi hút thuốc đến chi phí y tế là cực kỳ biến động và khó dự đoán hơn.



#### Kết luận rút ra cho bài phân tích:

* **Hút thuốc là "biến nhân" (multiplier):** Nó đẩy chi phí y tế lên một mức hoàn toàn khác, bất kể người đó đang ở độ tuổi nào.
* **Giá trị thực tiễn:** Biểu đồ này chứng minh rằng việc kết hợp hai biến `age` và `smoker` sẽ mang lại khả năng giải thích biến động của `charges` tốt hơn nhiều so với việc chỉ nhìn vào từng biến riêng lẻ.


### **Phân tích sự phân hóa chi phí (Boxplot)**

In [ ]:
plot_smoker_costs_boxplot(df_clean)

Để đi sâu vào cấu trúc phân phối của dữ liệu, ta sử dụng biểu đồ Boxplot so sánh biến `charges` dựa trên trạng thái `smoker`.

#### Các quan sát chủ đạo:
1. **Vị trí trung vị:** Có sự dịch chuyển rõ rệt về nền tảng chi phí. Nhóm đối tượng hút thuốc (1) có mức chi phí trung vị cao vượt trội, khẳng định kết quả từ kiểm định T-test về sự khác biệt có ý nghĩa thống kê.
2. **Biến thiên dữ liệu:** Nhóm hút thuốc có độ phân tán dữ liệu rộng hơn, phản ánh mức độ rủi ro và chi phí y tế khó dự đoán hơn so với nhóm còn lại.
3. **Các trường hợp đặc biệt (Outliers):** Nhóm không hút thuốc (0) ghi nhận nhiều giá trị ngoại lai có mức chi phí tương đương với nhóm hút thuốc. Điều này chứng tỏ dù không hút thuốc, các yếu tố rủi ro khác (như tuổi già hoặc bệnh nền) vẫn có thể đẩy chi phí lên cao, nhưng tần suất là không nhiều so với nhóm hút thuốc.

#### Kết luận:
Biểu đồ Boxplot một lần nữa xác nhận việc hút thuốc không chỉ làm tăng mức chi phí trung bình mà còn làm tăng mức độ biến động của chi phí y tế cá nhân.

### **Mối quan hệ giữa BMI, Hút thuốc và Chi phí y tế (Scatter Plot)**

In [ ]:
plot_bmi_vs_charges(df_clean)

Khi quan sát biểu đồ "Chi phí y tế vs BMI (theo trạng thái hút thuốc)", một bức tranh hoàn toàn khác hiện ra, giải thích tại sao hệ số tương quan của BMI lại thấp:

* **Nhóm không hút thuốc (smoker = 0 - Màu tím):** Chi phí y tế duy trì ở mức thấp (thường dưới 15,000) và khá ổn định, bất kể chỉ số BMI tăng hay giảm. Điều này kéo hệ số tương quan tổng thể của BMI xuống thấp.
* **Nhóm có hút thuốc (smoker = 1 - Màu cam):**
    * Có sự phân hóa cực kỳ rõ rệt tại ngưỡng **BMI ≈ 30**.
    * **BMI < 30:** Chi phí dao động ở mức cao (khoảng 15,000 - 30,000).
    * **BMI > 30 (Béo phì):** Chi phí "nhảy vọt" lên mức cao nhất (trên 30,000).
#### Kết luận:

1.  **Hiệu ứng tương tác (Interaction Effect):** BMI không tác động độc lập đến chi phí. Sức mạnh của nó chỉ được "kích hoạt" khi đi kèm với hành vi hút thuốc. Đây là lý do tại sao hệ số tương quan đơn thuần của BMI bị che lấp.
2.  **Ngưỡng rủi ro kép:** Một người vừa hút thuốc vừa có BMI > 30 đối mặt với rủi ro tài chính y tế cao gấp nhiều lần so với các nhóm còn lại.

---

## 📝 4. Kết luận

### 1. Tổng hợp Insight (Key Insights)

Sau quá trình phân tích kỹ lưỡng các yếu tố ảnh hưởng đến chi phí y tế, ba phát hiện cốt lõi đã được xác định:

* **Hành vi hút thuốc là biến số chi phối:** Đây là yếu tố dự báo mạnh nhất về sự gia tăng đột biến của chi phí. Người hút thuốc có mức chi trả trung bình cao hơn rõ rệt so với nhóm còn lại, bất kể các yếu tố khác.
* **Hiệu ứng cộng hưởng "BMI - Hút thuốc":** Một insight quan trọng bị che lấp trong ma trận tương quan đơn thuần là sự tương tác giữa BMI và thuốc lá. Chỉ số BMI cao (trên 30) chỉ thực sự trở thành "gánh nặng tài chính" khi đi kèm với hành vi hút thuốc. Ngược lại, đối với nhóm không hút thuốc, BMI không tạo ra sự biến động chi phí quá lớn.
* **Tác động của quá trình lão hóa:** Tuổi tác có mối tương quan thuận ổn định với chi phí, phản ánh quy luật sinh học về nhu cầu chăm sóc sức khỏe tăng dần theo thời gian. Các yếu tố như giới tính và vùng miền trong tập dữ liệu này không cho thấy sự phân hóa đáng kể.

### 2. Giá trị thực tiễn (Practical Value)

Những kết quả phân tích này mang lại giá trị ứng dụng cụ thể trong quản trị rủi ro và chiến lược kinh doanh:

* **Cá nhân hóa phí bảo hiểm:** Công ty có thể xây dựng mô hình định giá phí bảo hiểm chính xác hơn bằng cách tập trung trọng số vào nhóm "nguy cơ kép" (hút thuốc và béo phì) thay vì chỉ nhìn vào BMI đơn lẻ.
* **Chiến dịch can thiệp sức khỏe mục tiêu:** Thay vì triển khai các chương trình giảm cân đại trà, tổ chức có thể tối ưu hóa nguồn lực bằng cách tập trung hỗ trợ cai thuốc cho nhóm có chỉ số BMI cao, từ đó kỳ vọng **giảm thiểu mức chi trả bồi thường bảo hiểm từ 15-20%** cho phân khúc rủi ro này.
* **Sàng lọc khách hàng:** Giúp đội ngũ thẩm định ưu tiên các chỉ số quan trọng (Age, Smoker, BMI) để đơn giản hóa quy trình đánh giá nhưng vẫn đảm bảo tính chính xác.

### 3. Hạn chế và Hướng phát triển (Limitations & Future Work)

 **Hạn chế:**
* **Thiếu hụt dữ liệu thói quen:** Tập dữ liệu hiện tại chưa bao gồm các thông tin quan trọng như tiền sử bệnh lý gia đình, chế độ dinh dưỡng, và cường độ tập luyện thể thao – những yếu tố có thể giải thích phần sai số còn lại của chi phí.
* **Tính tĩnh của dữ liệu:** Dữ liệu chỉ phản ánh một thời điểm nhất định (cross-sectional), chưa cho thấy sự thay đổi của chi phí theo lộ trình điều trị dài hạn của từng cá nhân.


 **Hướng phát triển:**
* **Bổ sung tính năng:** Trong các phiên bản sau, cần thu thập thêm dữ liệu về các bệnh mãn tính sẵn có (tiểu đường, huyết áp) để tăng cường độ sâu cho insight.
* **Phân tích phi tuyến tính:** Áp dụng các kỹ thuật chuyển đổi biến (feature transformation) hoặc xem xét các mô hình tương tác để khai thác triệt để mối quan hệ giữa BMI và thuốc lá mà các phép đo tuyến tính đơn thuần dễ bỏ sót.
* **Mở rộng mẫu:** Thu thập thêm dữ liệu từ các khu vực địa lý khác nhau để kiểm chứng tính phổ quát của các nhận định trên quy mô lớn hơn.